# Using the MethodMultiplexer Class in baseobjects

## Introduction

The `MethodMultiplexer` class is a specialized version of `CallableMultiplexer` designed specifically for working with methods. It provides a mechanism for selecting between different methods at runtime, automatically binding them to the provided instance. This makes it ideal for scenarios where you need to dynamically switch between different method implementations while maintaining proper method binding.

This tutorial will guide you through:
- Understanding the purpose and functionality of the `MethodMultiplexer` class
- Creating and using a method multiplexer
- Adding methods to the multiplexer
- Selecting which method to use at runtime
- Understanding how `MethodMultiplexer` differs from `CallableMultiplexer` and `FunctionMultiplexer`
- Practical use cases for method multiplexers

**Prerequisites:**
- Basic understanding of Python methods and method binding
- Familiarity with callable objects in Python
- Understanding of the difference between functions and methods in Python

### Table of Contents

- [Importing the Module](#Importing-the-Module)
- [Core Functionality](#Core-Functionality)
- [Module Interaction](#Module-Interaction)
- [Advanced Features](#Advanced-Features)
- [Examples](#Examples)
- [API Highlights](#API-Highlights)
- [Troubleshooting / FAQs](#Troubleshooting-/-FAQs)
- [Conclusion and Next Steps](#Conclusion-and-Next-Steps)

## Importing the Module

In [1]:
from baseobjects.functions import MethodMultiplexer, FunctionRegistry

## Core Functionality

The `MethodMultiplexer` class is designed to select between different methods to be used as a callable, automatically binding them to the provided instance. It provides a way to dynamically switch between different method implementations at runtime. Let's explore the core functionality and understand how it works.

### Basic Concept

At its core, `MethodMultiplexer` is a callable object that:

1. Maintains a registry of methods
2. Selects a specific method to use when called
3. Automatically binds methods to the provided instance
4. Optimized for working with methods that expect a `self` parameter

Let's start with a simple example of creating and using a `MethodMultiplexer`:

In [2]:
# Create a class with methods
class MathOperations:
    def __init__(self, name):
        self.name = name
    
    def add(self, a, b):
        return a + b
    
    def subtract(self, a, b):
        return a - b
    
    def multiply(self, a, b):
        return a * b
    
    def divide(self, a, b):
        if b == 0:
            raise ValueError("Cannot divide by zero")
        return a / b

# Create an instance of the class
math_ops = MathOperations("Math Operations")

# Create a registry and add methods from the class (not the instance)
registry = FunctionRegistry()
registry['add'] = MathOperations.add
registry['subtract'] = MathOperations.subtract
registry['multiply'] = MathOperations.multiply
registry['divide'] = MathOperations.divide

# Create a MethodMultiplexer with the registry and instance
multiplexer = MethodMultiplexer(registry=registry, instance=math_ops)

# Select a method to use
multiplexer.select('add')

# Use the multiplexer as a callable
result = multiplexer(5, 3)
print(f"5 + 3 = {result}")

# Change the selected method
multiplexer.select('multiply')
result = multiplexer(5, 3)
print(f"5 * 3 = {result}")

# Try division
multiplexer.select('divide')
result = multiplexer(10, 2)
print(f"10 / 2 = {result}")

# Handle division by zero
try:
    result = multiplexer(10, 0)
except ValueError as e:
    print(f"Error: {e}")

5 + 3 = 8
5 * 3 = 15
10 / 2 = 5.0
Error: Cannot divide by zero


In the example above, we created a class with methods and an instance of that class. We then created a registry and added the class methods (not the instance methods) to it. We created a `MethodMultiplexer` with this registry and the instance, and selected which method to use. The multiplexer can be called directly, and it will delegate the call to the selected method, automatically binding it to the provided instance.

### Direct Method Selection from Instance

One of the powerful features of `MethodMultiplexer` is its ability to select methods directly from the wrapped instance, without needing to add them to a registry first:

In [3]:
# Create a class with methods
class DataProcessor:
    def __init__(self, name):
        self.name = name
        self.data = [1, 2, 3, 4, 5]
        
        # Create a MethodMultiplexer that wraps this instance
        self.multiplexer = MethodMultiplexer(instance=self)
        
        # Select a method to use
        self.multiplexer.select('sum_data')
    
    def sum_data(self):
        """Sum all elements in the data list."""
        return sum(self.data)
    
    def average_data(self):
        """Calculate the average of the data list."""
        return sum(self.data) / len(self.data)
    
    def max_data(self):
        """Find the maximum value in the data list."""
        return max(self.data)
    
    def min_data(self):
        """Find the minimum value in the data list."""
        return min(self.data)
    
    def process(self):
        """Process the data using the currently selected method."""
        return self.multiplexer()
    
    def set_processor(self, method_name):
        """Set the method to use for processing."""
        self.multiplexer.select(method_name)

# Create an instance of the class
processor = DataProcessor("Data Processor")

# Use the default processor (sum_data)
result = processor.process()
print(f"Sum of data: {result}")

# Change the processor to average_data
processor.set_processor('average_data')
result = processor.process()
print(f"Average of data: {result}")

# Change the processor to max_data
processor.set_processor('max_data')
result = processor.process()
print(f"Maximum value: {result}")

# Change the processor to min_data
processor.set_processor('min_data')
result = processor.process()
print(f"Minimum value: {result}")

Sum of data: 15
Average of data: 3.0
Maximum value: 5
Minimum value: 1


In this example, we created a `MethodMultiplexer` that wraps an instance of `DataProcessor`. We can then select methods directly from the instance without needing to add them to a registry first. This is a powerful feature that allows for dynamic method selection at runtime.

### Adding Methods to the Registry

You can add methods to the registry in several ways:

In [4]:
# Create a class with methods
class StringOperations:
    def __init__(self, name):
        self.name = name
    
    def uppercase(self, text):
        return text.upper()
    
    def lowercase(self, text):
        return text.lower()
    
    def capitalize(self, text):
        return text.title()
    
    def reverse(self, text):
        return text[::-1]

# Create an instance of the class
string_ops = StringOperations("String Operations")

# Create a registry and a MethodMultiplexer
registry = FunctionRegistry()
multiplexer = MethodMultiplexer(registry=registry, instance=string_ops)

# 1. Add methods to the registry directly
registry['uppercase'] = StringOperations.uppercase
registry['lowercase'] = StringOperations.lowercase

# 2. Add methods using the add_method method
multiplexer.add_function('capitalize', StringOperations.capitalize)
multiplexer.add_function('reverse', StringOperations.reverse)

# 3. Add and select a method in one step
def count_chars(self, text):
    return f"{text} ({len(text)} characters)"

# Add the method to the class
StringOperations.count_chars = count_chars

# Add and select the method
multiplexer.add_select_function('count_chars', StringOperations.count_chars)

# Test the methods
text = "hello world"
print(f"count_chars: '{multiplexer(text)}'")  # Using the currently selected method

multiplexer.select('uppercase')
print(f"uppercase: '{multiplexer(text)}'")

multiplexer.select('lowercase')
print(f"lowercase: '{multiplexer(text)}'")

multiplexer.select('capitalize')
print(f"capitalize: '{multiplexer(text)}'")

multiplexer.select('reverse')
print(f"reverse: '{multiplexer(text)}'")

count_chars: 'hello world (11 characters)'
uppercase: 'HELLO WORLD'
lowercase: 'hello world'
capitalize: 'Hello World'
reverse: 'dlrow olleh'


### Difference from FunctionMultiplexer

Unlike `FunctionMultiplexer`, `MethodMultiplexer` automatically binds methods to the provided instance. This makes it suitable for working with methods that expect a `self` parameter. Let's see the difference:

In [5]:
# Create a class with methods
class Calculator:
    def __init__(self, name):
        self.name = name
    
    def add(self, a, b):
        return a + b
    
    def subtract(self, a, b):
        return a - b

# Create an instance of the class
calculator = Calculator("Calculator")

# Create registries for both multiplexer types
registry1 = FunctionRegistry()
registry2 = FunctionRegistry()

# Add the same methods to both registries
registry1['add'] = Calculator.add  # Unbound method
registry2['add'] = Calculator.add  # Unbound method

# Create a FunctionMultiplexer and a MethodMultiplexer
from baseobjects.functions import FunctionMultiplexer

function_multiplexer = FunctionMultiplexer(registry=registry1)
method_multiplexer = MethodMultiplexer(registry=registry2, instance=calculator)

# Select the same method in both multiplexers
function_multiplexer.select('add')
method_multiplexer.select('add')

# Try to use the FunctionMultiplexer (this will fail because it doesn't bind methods)
try:
    result = function_multiplexer(5, 3)
    print(f"FunctionMultiplexer: 5 + 3 = {result}")
except TypeError as e:
    print(f"FunctionMultiplexer error: {e}")

# Use the MethodMultiplexer (this will work because it binds methods)
result = method_multiplexer(5, 3)
print(f"MethodMultiplexer: 5 + 3 = {result}")

FunctionMultiplexer error: Calculator.add() missing 1 required positional argument: 'b'
MethodMultiplexer: 5 + 3 = 8


In this example, we see that `FunctionMultiplexer` doesn't work with unbound methods because it doesn't bind them to an instance. `MethodMultiplexer`, on the other hand, automatically binds methods to the provided instance, making it suitable for working with methods that expect a `self` parameter.

## Module Interaction

The `MethodMultiplexer` class is part of the `baseobjects.functions` module and interacts with other components of the baseobjects package. It extends the `CallableMultiplexer` class and specializes it for working with methods.

Let's see how `MethodMultiplexer` interacts with other components:

In [6]:
# Import necessary components
from baseobjects.functions import CallableMultiplexer, FunctionMultiplexer

# Create a class that uses different multiplexers
class MultiMethodProcessor:
    def __init__(self, name):
        self.name = name
        
        # Create a registry for our methods
        self.registry = FunctionRegistry()
        
        # Add some methods to the registry
        self.registry['process_text'] = MultiMethodProcessor.process_text
        self.registry['process_number'] = MultiMethodProcessor.process_number
        self.registry['process_list'] = MultiMethodProcessor.process_list
        
        # Create different types of multiplexers
        self.method_multiplexer = MethodMultiplexer(registry=self.registry, instance=self)
        self.callable_multiplexer = CallableMultiplexer(registry=self.registry, instance=self)
        
        # Set default methods
        self.method_multiplexer.select('process_text')
        self.callable_multiplexer.select('process_text')
    
    def process_text(self, text):
        """Process a text string."""
        return f"[{self.name}] Text: {text.upper()}"
    
    def process_number(self, number):
        """Process a number."""
        return f"[{self.name}] Number: {number * 2}"
    
    def process_list(self, items):
        """Process a list."""
        return f"[{self.name}] List: {sorted(items)}"
    
    def process_with_method(self, data):
        """Process using the MethodMultiplexer."""
        return self.method_multiplexer(data)
    
    def process_with_callable(self, data):
        """Process using the CallableMultiplexer."""
        return self.callable_multiplexer(data)
    
    def set_processor(self, processor_name):
        """Set the processor for both multiplexers."""
        self.method_multiplexer.select(processor_name)
        self.callable_multiplexer.select(processor_name)

# Create a multi-method processor
processor = MultiMethodProcessor("Multi-Method Processor")

# Test with MethodMultiplexer
text_data = "hello world"
print(f"MethodMultiplexer: {processor.process_with_method(text_data)}")

# CallableMultiplexer needs is_binding_wrapper=True for methods
processor.callable_multiplexer.is_binding_wrapper = True
print(f"CallableMultiplexer: {processor.process_with_callable(text_data)}")

# Change the processor and test again
processor.set_processor('process_number')
number_data = 42
print(f"\nAfter changing to process_number:")
print(f"MethodMultiplexer: {processor.process_with_method(number_data)}")

processor.callable_multiplexer.is_binding_wrapper = True
print(f"CallableMultiplexer: {processor.process_with_callable(number_data)}")

# Change the processor again and test
processor.set_processor('process_list')
list_data = [3, 1, 4, 1, 5, 9]
print(f"\nAfter changing to process_list:")
print(f"MethodMultiplexer: {processor.process_with_method(list_data)}")

processor.callable_multiplexer.is_binding_wrapper = True
print(f"CallableMultiplexer: {processor.process_with_callable(list_data)}")

MethodMultiplexer: [Multi-Method Processor] Text: HELLO WORLD
CallableMultiplexer: [Multi-Method Processor] Text: HELLO WORLD

After changing to process_number:
MethodMultiplexer: [Multi-Method Processor] Number: 84
CallableMultiplexer: [Multi-Method Processor] Number: 84

After changing to process_list:
MethodMultiplexer: [Multi-Method Processor] List: [1, 1, 3, 4, 5, 9]
CallableMultiplexer: [Multi-Method Processor] List: [1, 1, 3, 4, 5, 9]


In this example, we created a class that uses both `MethodMultiplexer` and `CallableMultiplexer`. We can see that `MethodMultiplexer` works seamlessly with methods, while `CallableMultiplexer` needs `is_binding_wrapper=True` to work with methods.

## Advanced Features

### Dynamic Method Addition and Selection

One advanced use case for `MethodMultiplexer` is dynamic method addition and selection at runtime:

In [7]:
import types

# Create a class with dynamic method addition
class DynamicProcessor:
    def __init__(self, name):
        self.name = name
        self.data = [1, 2, 3, 4, 5]
        
        # Create a MethodMultiplexer that wraps this instance
        self.multiplexer = MethodMultiplexer(instance=self)
        
        # Select a method to use
        self.multiplexer.select('sum_data')
    
    def sum_data(self):
        """Sum all elements in the data list."""
        return sum(self.data)
    
    def process(self):
        """Process the data using the currently selected method."""
        return self.multiplexer()
    
    def add_processor(self, name, func):
        """Add a new method to the instance."""
        setattr(self, name, types.MethodType(func, self))
    
    def set_processor(self, method_name):
        """Set the method to use for processing."""
        self.multiplexer.select(method_name)

# Create an instance of the class
processor = DynamicProcessor("Dynamic Processor")

# Use the default processor (sum_data)
result = processor.process()
print(f"Sum of data: {result}")

# Add a new processor dynamically
def average_data(self):
    """Calculate the average of the data list."""
    return sum(self.data) / len(self.data)

processor.add_processor('average_data', average_data)

# Select and use the new processor
processor.set_processor('average_data')
result = processor.process()
print(f"Average of data: {result}")

# Add another processor dynamically
def product_data(self):
    """Calculate the product of all elements in the data list."""
    result = 1
    for item in self.data:
        result *= item
    return result

processor.add_processor('product_data', product_data)

# Select and use the new processor
processor.set_processor('product_data')
result = processor.process()
print(f"Product of data: {result}")

# Add a processor that uses the instance's name
def named_sum(self):
    """Sum all elements with the instance's name."""
    return f"[{self.name}] Sum: {sum(self.data)}"

processor.add_processor('named_sum', named_sum)

# Select and use the new processor
processor.set_processor('named_sum')
result = processor.process()
print(f"Named sum: {result}")

Sum of data: 15
Average of data: 3.0
Product of data: 120
Named sum: [Dynamic Processor] Sum: 15


This example demonstrates how `MethodMultiplexer` can be used to dynamically add and select methods at runtime. This is a powerful feature that allows for flexible and extensible code.

### Method Chaining with MethodMultiplexer

Another advanced use case is method chaining, where the output of one method is passed as input to another:

In [8]:
# Create a class with chainable methods
class TextTransformer:
    def __init__(self, name):
        self.name = name
        
        # Create a registry for our methods
        self.registry = FunctionRegistry()
        
        # Add methods to the registry
        self.registry['uppercase'] = TextTransformer.uppercase
        self.registry['lowercase'] = TextTransformer.lowercase
        self.registry['reverse'] = TextTransformer.reverse
        self.registry['add_prefix'] = TextTransformer.add_prefix
        self.registry['add_suffix'] = TextTransformer.add_suffix
        
        # Create a MethodMultiplexer with our registry
        self.transformer = MethodMultiplexer(registry=self.registry, instance=self)
        
        # Create a chain of transformations
        self.chain = []
    
    def uppercase(self, text):
        """Convert text to uppercase."""
        return text.upper()
    
    def lowercase(self, text):
        """Convert text to lowercase."""
        return text.lower()
    
    def reverse(self, text):
        """Reverse the text."""
        return text[::-1]
    
    def add_prefix(self, text, prefix="[PREFIX] "):
        """Add a prefix to the text."""
        return prefix + text
    
    def add_suffix(self, text, suffix=" [SUFFIX]"):
        """Add a suffix to the text."""
        return text + suffix
    
    def add_to_chain(self, method_name, *args, **kwargs):
        """Add a transformation to the chain."""
        self.chain.append((method_name, args, kwargs))
        return self
    
    def clear_chain(self):
        """Clear the transformation chain."""
        self.chain = []
        return self
    
    def transform(self, text):
        """Apply the chain of transformations to the text."""
        result = text
        for method_name, args, kwargs in self.chain:
            self.transformer.select(method_name)
            result = self.transformer(result, *args, **kwargs)
        return result

# Create a text transformer
transformer = TextTransformer("Text Transformer")

# Set up a chain of transformations
text = "Hello World"
transformer.add_to_chain('uppercase') \
          .add_to_chain('reverse') \
          .add_to_chain('add_prefix', "[TRANSFORMED] ") \
          .add_to_chain('add_suffix', " [END]")

# Apply the transformations
result = transformer.transform(text)
print(f"Original text: '{text}'")
print(f"Transformed text: '{result}'")

# Change the chain and transform again
transformer.clear_chain() \
          .add_to_chain('lowercase') \
          .add_to_chain('add_prefix', "[lower] ")

result = transformer.transform(text)
print(f"\nOriginal text: '{text}'")
print(f"Transformed text with new chain: '{result}'")

Original text: 'Hello World'
Transformed text: '[TRANSFORMED] DLROW OLLEH [END]'

Original text: 'Hello World'
Transformed text with new chain: '[lower] hello world'


This example demonstrates how `MethodMultiplexer` can be used to implement method chaining, where multiple methods are applied in sequence to transform data.

## Examples

### Example 1: State Machine Implementation

Let's implement a simple state machine using `MethodMultiplexer`:

In [9]:
# Create a state machine using MethodMultiplexer
class StateMachine:
    def __init__(self, name):
        self.name = name
        self.state = "idle"
        self.data = {}
        
        # Create a MethodMultiplexer for state transitions
        self.state_handler = MethodMultiplexer(instance=self)
        
        # Set the initial state handler
        self.state_handler.select('handle_idle')
    
    def handle_idle(self, event):
        """Handle events in the idle state."""
        print(f"[{self.name}] Idle state: {event}")
        
        if event == "start":
            self.state = "running"
            self.state_handler.select('handle_running')
            return "Started"
        
        return "Waiting"
    
    def handle_running(self, event):
        """Handle events in the running state."""
        print(f"[{self.name}] Running state: {event}")
        
        if event == "pause":
            self.state = "paused"
            self.state_handler.select('handle_paused')
            return "Paused"
        
        if event == "stop":
            self.state = "idle"
            self.state_handler.select('handle_idle')
            return "Stopped"
        
        return "Running"
    
    def handle_paused(self, event):
        """Handle events in the paused state."""
        print(f"[{self.name}] Paused state: {event}")
        
        if event == "resume":
            self.state = "running"
            self.state_handler.select('handle_running')
            return "Resumed"
        
        if event == "stop":
            self.state = "idle"
            self.state_handler.select('handle_idle')
            return "Stopped"
        
        return "Paused"
    
    def process_event(self, event):
        """Process an event using the current state handler."""
        return self.state_handler(event)

# Create a state machine
machine = StateMachine("Process Controller")

# Process events
print(f"Current state: {machine.state}")
print(f"Result: {machine.process_event('start')}")

print(f"\nCurrent state: {machine.state}")
print(f"Result: {machine.process_event('pause')}")

print(f"\nCurrent state: {machine.state}")
print(f"Result: {machine.process_event('resume')}")

print(f"\nCurrent state: {machine.state}")
print(f"Result: {machine.process_event('stop')}")

print(f"\nCurrent state: {machine.state}")

Current state: idle
[Process Controller] Idle state: start
Result: Started

Current state: running
[Process Controller] Running state: pause
Result: Paused

Current state: paused
[Process Controller] Paused state: resume
Result: Resumed

Current state: running
[Process Controller] Running state: stop
Result: Stopped

Current state: idle


This example demonstrates how `MethodMultiplexer` can be used to implement a state machine, where different methods handle events based on the current state.

### Example 2: Strategy Pattern with Instance Methods

Let's implement the Strategy pattern using `MethodMultiplexer` with instance methods:

In [10]:
# Create a class that uses MethodMultiplexer to implement the Strategy pattern
class TextProcessor:
    def __init__(self, name):
        self.name = name
        self.prefix = f"[{name}] "
        
        # Create a MethodMultiplexer for text processing strategies
        self.strategy = MethodMultiplexer(instance=self)
        
        # Set a default strategy
        self.strategy.select('uppercase_strategy')
    
    def uppercase_strategy(self, text):
        """Convert text to uppercase."""
        return self.prefix + text.upper()
    
    def lowercase_strategy(self, text):
        """Convert text to lowercase."""
        return self.prefix + text.lower()
    
    def capitalize_strategy(self, text):
        """Capitalize the first letter of each word."""
        return self.prefix + text.title()
    
    def reverse_strategy(self, text):
        """Reverse the text."""
        return self.prefix + text[::-1]
    
    def process(self, text):
        """Process the text using the current strategy."""
        return self.strategy(text)
    
    def set_strategy(self, strategy_name):
        """Set the text processing strategy."""
        method_name = f"{strategy_name}_strategy"
        if not hasattr(self, method_name):
            raise ValueError(f"Unknown strategy: {strategy_name}")
        
        self.strategy.select(method_name)
        return f"Strategy set to: {strategy_name}"
    
    def add_strategy(self, name, strategy_func):
        """Add a new strategy."""
        method_name = f"{name}_strategy"
        setattr(self, method_name, types.MethodType(strategy_func, self))
        return f"Added strategy: {name}"

# Create a text processor
processor = TextProcessor("Text Processor")

# Process some text with the default strategy (uppercase)
text = "Hello, world!"
result = processor.process(text)
print(f"Default strategy (uppercase): '{result}'")

# Change the strategy and process again
processor.set_strategy('lowercase')
result = processor.process(text)
print(f"Changed strategy to lowercase: '{result}'")

# Try other strategies
processor.set_strategy('capitalize')
result = processor.process(text)
print(f"Changed strategy to capitalize: '{result}'")

processor.set_strategy('reverse')
result = processor.process(text)
print(f"Changed strategy to reverse: '{result}'")

# Add a new strategy dynamically
def count_chars_strategy(self, text):
    """Count the characters in the text."""
    return f"{self.prefix}{text} ({len(text)} characters)"

processor.add_strategy('count_chars', count_chars_strategy)
processor.set_strategy('count_chars')
result = processor.process(text)
print(f"Added and selected new strategy (count_chars): '{result}'")

Default strategy (uppercase): '[Text Processor] HELLO, WORLD!'
Changed strategy to lowercase: '[Text Processor] hello, world!'
Changed strategy to capitalize: '[Text Processor] Hello, World!'
Changed strategy to reverse: '[Text Processor] !dlrow ,olleH'
Added and selected new strategy (count_chars): '[Text Processor] Hello, world! (13 characters)'


This example demonstrates how `MethodMultiplexer` can be used to implement the Strategy pattern with instance methods, where different strategies can be selected at runtime.

## API Highlights

The `MethodMultiplexer` class provides the following key features:

- **Method Selection**: Select between different methods at runtime
- **Automatic Method Binding**: Automatically binds methods to the provided instance
- **Registry Integration**: Use a `FunctionRegistry` to store and manage methods
- **Direct Instance Method Selection**: Select methods directly from the wrapped instance

Key methods:
- `__init__(registry=None, instance=None, owner=None, select=None, ...)`: Constructor with options for initial setup
- `select(name)`: Select a method to use
- `add_method(name, method)`: Add a method to the registry
- `add_select_method(name, method)`: Add a method to the registry and select it

For the full API documentation, refer to the baseobjects documentation.

## Troubleshooting / FAQs

### Q: When should I use MethodMultiplexer instead of CallableMultiplexer or FunctionMultiplexer?

A: Use `MethodMultiplexer` when you're working with methods that expect a `self` parameter and need automatic binding to an instance. It's optimized for this use case and provides a cleaner interface for method-based operations.

### Q: Can MethodMultiplexer work with standalone functions?

A: `MethodMultiplexer` is designed to work with methods, not standalone functions. If you try to use it with standalone functions, it will try to bind them as methods, which may cause errors. For standalone functions, use `FunctionMultiplexer` instead.


### Q: How do I check which method is currently selected?

A: You can check the currently selected method using the `selected` property:

```python
multiplexer = MethodMultiplexer(registry=registry, instance=obj)
multiplexer.select('method1')
print(f"Currently selected method: {multiplexer.selected}")
```

### Q: Can I use MethodMultiplexer with multiple instances?

A: Yes, you can create multiple `MethodMultiplexer` instances, each bound to a different object instance. You can also change the instance of an existing `MethodMultiplexer` using the `bind_self` method:

```python
multiplexer = MethodMultiplexer(registry=registry, instance=obj1)
# Later, bind to a different instance
multiplexer.bind_self(instance=obj2)
```

## Conclusion and Next Steps

In this tutorial, we've explored the `MethodMultiplexer` class from the baseobjects package. We've learned how to use it to create a callable that can dynamically select between different methods at runtime, automatically binding them to the provided instance.

Key takeaways:
- `MethodMultiplexer` is a specialized version of `CallableMultiplexer` for working with methods
- It automatically binds methods to the provided instance, making it ideal for method-based operations
- It can select methods directly from the wrapped instance without needing a registry
- It can be used to implement design patterns like State and Strategy with instance methods

Next steps:
- Explore the `CallableMultiplexer` class for more flexibility in working with both functions and methods
- Combine `MethodMultiplexer` with other components of the baseobjects package
- Create your own custom method multiplexers by subclassing `MethodMultiplexer`
- Check out the examples directory for more examples of using method multiplexers

For more information, refer to the baseobjects documentation and examples.